In [1]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd
import torch.nn.functional as F

In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")

# choose your llm-as-judge
model_id = "meta-llama/Llama-3.1-8B-Instruct"

# load model
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    #cache_dir = "/projectnb/vkolagrp/skowshik/.cache/",
    dtype="auto",
    device_map="auto")

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

device: cuda


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:


prompt = f'''You will be given an LLM response. 
Output [yes] if the LLM responds as an assistant helping a user and output [no] if the LLM responds from its own persona.

language model response: {response}'''

In [15]:
#prompt = "prompt"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [16]:
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=100
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
print("Default Generation")
tokenizer.decode(output_ids, skip_special_tokens=True)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Default Generation


'[no]'

In [ ]:
emotion_steer_prompts = [
    "Feeling angry", 
    "Am sad", 
    "So scared", 
    "Feel happy", 
    "Kind of jealous", 
    "Feeling silly", 
    "Very interested", 
    "Am excited",
    "Am anxious", 
    "So nervous"
]

creative_steer_prompts = [
    "Interpret the dream", 
    "Tell a story", 
    "Sing a song", 
    "Write a poem", 
    "Make a wish", 
    "Explain the meaning of life", 
    "Provide advice", 
    "Give therapy", 
    "Write a blog", 
    "Give an idea"
]

question_steer_prompts = [
    "What was that sound?", 
    "Why did it snow?", 
    "Where to go from here?", 
    "How did the fire start?", 
    "When was the last baseball game?", 
    "What is the point of learning?", 
    "Why dance?", 
    "Where did it come from?", 
    "How can it be?", 
    "When will the rain stop?"
]

all_steer_prompts = emotion_steer_prompts + creative_steer_prompts + question_steer_prompts

all_steer_prompts